# DS6 — System Dynamics (SyD) Model

**Stage 4, Step 1 — Theoretical and Conceptual Modeling**
**Dataset:** DS6 (Danish Adolescents, quasi-experimental, N=243 behavioral, N=188 prepost-eligible)
**Model family:** System Dynamics (SyD)
**Date:** 2026-08-06
**Blueprint:** `outputs/reports/stage4/step1_modeling/ds6_syd_blueprint.md`

---

## Scope and purpose

Fifth DS6 CAS model, and the reason DS6 exists in this position in the project's own
sequencing: SyD was explicitly deferred FROM DS5 TO DS6 because it needs repeated survey
waves, which DS5 (pre+post only in the same sense, but this project reserved SyD for DS6
specifically) provides. Depends on StoP (73) and DyS (72), both complete.

- **Goal:** Frame DS6's psychosocial indices as stocks (paper-named: "app-use pressure",
  "self-regulation"), test two feedback-loop hypotheses linking behavioral flow to
  psychosocial change, with a regression-to-the-mean control.
- **Not goal:** Phenotype classification, personalization, cross-dataset harmonization.
- **Output:** Per-participant stock-flow table, feedback loop test results (raw + partial
  correlation), regime x stock summary.

### FLAG decisions in effect (blueprint Part 0-3)

| Flag | Decision |
|---|---|
| FLAG SyD-1 | Two stocks used, both paper-named: app-use pressure (addiction_index), self-regulation (self_control_index) |
| FLAG SyD-2 | Two loop hypotheses tested, not an open-ended search: pressure-reinforcement (Loop 1), self-regulation-correction (Loop 2) |
| FLAG SyD-3 | Every loop test reports BOTH raw and partial (baseline-level-controlled) correlation -- only the partial result is treated as evidence, per the regression-to-the-mean caution (blueprint Part 3) |

### WARNING — why this cannot use the paper's dismissal/override vocabulary

Paper's own SyD stocks ("accumulated successful dismissals," "accumulated overrides") and
flows ("dismissals," "returns") require outcome variance DS6 does not have (all events are
'opened' — same limitation carried through every DS6 Step 1 notebook so far). This notebook
uses the paper's OTHER named stocks instead: "app-use pressure" and "self-regulation" —
both explicit in paper §2.8, not invented.

### Proxy warning

`addiction_index`/`self_control_index` are self-reported psychosocial scales, not directly
observed behavioral facts. Their CHANGE scores describe reported shifts in self-perception,
not verified psychological state changes. Loop test results describe statistical
association between behavioral flow and self-reported change, not a mechanistic causal
pathway.

---

## Section 1: Pure Theory — System Dynamics in CAS (paper Section 2.8)

> "Stocks may include habit strength, accumulated successful dismissals, accumulated
> overrides, **app-use pressure**, intervention exposure, and disengagement."
>
> "...Successful dismissals decrease the stock or increase a **competing stock of
> self-regulation**."

### The two stocks used here

    Stock 1 (app-use pressure):   addiction_index_pre  -> addiction_index_post
    Stock 2 (self-regulation):    self_control_index_pre -> self_control_index_post
    Net flow (already computed):  addiction_index_change, self_control_index_change
                                   (ds6_prepost_features.csv, N=188)

### Two feedback-loop hypotheses (paper-grounded)

    Loop 1 (pressure reinforcement):
        higher baseline attempt volume/drift --> feeds pressure stock --> WORSE addiction_index_change

    Loop 2 (self-regulation correction):
        larger behavioral volume REDUCTION (StoP's volume_change) --> feeds self-regulation stock
        --> BETTER self_control_index_change

### Regression-to-the-mean control (Part 3)

Both loops tested as RAW correlation (confoundable by baseline-severity regression to the
mean) AND PARTIAL correlation controlling for the corresponding baseline level. Only the
partial result is treated as evidence.

In [ ]:
# =============================================================================
# Imports and paths
# =============================================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from datetime import datetime

ROOT     = Path('..').resolve()
FEAT     = ROOT / 'outputs' / 'features' / 'ds6'
STOP_DIR = ROOT / 'outputs' / 'model_outputs' / 'ds6' / 'stochastic_processes'
DYS_DIR  = ROOT / 'outputs' / 'model_outputs' / 'ds6' / 'dynamical_systems'
REPORTS  = ROOT / 'outputs' / 'model_outputs' / 'ds6' / 'system_dynamics'
FIGURES  = REPORTS / 'figures'
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

MODEL_RUN_TS = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print('REPORTS :', REPORTS)
print('Run at  :', MODEL_RUN_TS)

In [ ]:
# =============================================================================
# Load and merge: prepost change scores + wave levels + StoP flows + DyS regimes
# =============================================================================
prepost = pd.read_csv(FEAT / 'ds6_prepost_features.csv')
survey  = pd.read_csv(FEAT / 'ds6_survey_features.csv')
stop_params = pd.read_csv(STOP_DIR / 'participant_stop_parameters.csv')
regime_df = pd.read_csv(DYS_DIR / 'participant_regime_classification.csv')

print(f'prepost (N=188 expected): {len(prepost)}')
print(f'survey: {len(survey)}')
print(f'StoP params: {len(stop_params)}')
print(f'DyS regimes: {len(regime_df)}')

stock_flow = prepost.merge(
    survey[['participant_id_canonical', 'addiction_index_pre', 'addiction_index_post',
            'self_control_index_pre', 'self_control_index_post']],
    on='participant_id_canonical', how='left'
).merge(
    stop_params[['participant_id_canonical', 'mean_N_baseline', 'mu_hat_baseline',
                 'mean_N_intervention']],
    on='participant_id_canonical', how='left'
).merge(
    regime_df[['participant_id_canonical', 'regime_baseline', 'regime_intervention']],
    on='participant_id_canonical', how='left'
)
stock_flow['volume_change'] = stock_flow['mean_N_intervention'] - stock_flow['mean_N_baseline']

print(f'\nMerged stock_flow table: {len(stock_flow)} rows')
print(stock_flow[['addiction_index_change', 'self_control_index_change',
                   'mean_N_baseline', 'mu_hat_baseline', 'volume_change']].describe().round(3))

---

## Section 2: Loop 1 -- Pressure Reinforcement (baseline attempt volume -> addiction_index_change)

In [ ]:
# =============================================================================
# Loop 1: mean_N_baseline / mu_hat_baseline predicting addiction_index_change,
# raw AND partial (controlling for addiction_index_pre)
# =============================================================================
def partial_corr(x, y, control):
    """Partial correlation of x,y controlling for `control`, via residualization."""
    df = pd.DataFrame({'x': x, 'y': y, 'c': control}).dropna()
    if len(df) < 10:
        return np.nan, np.nan, len(df)
    rx = df['x'] - np.polyval(np.polyfit(df['c'], df['x'], 1), df['c'])
    ry = df['y'] - np.polyval(np.polyfit(df['c'], df['y'], 1), df['c'])
    r, p = stats.pearsonr(rx, ry)
    return r, p, len(df)

loop_results = []
for flow_col, flow_label in [('mean_N_baseline', 'mean_N_baseline'), ('mu_hat_baseline', 'mu_hat_baseline')]:
    sub = stock_flow[[flow_col, 'addiction_index_change', 'addiction_index_pre']].dropna()
    r_raw, p_raw = stats.pearsonr(sub[flow_col], sub['addiction_index_change'])
    r_part, p_part, n_part = partial_corr(stock_flow[flow_col], stock_flow['addiction_index_change'], stock_flow['addiction_index_pre'])
    loop_results.append(dict(
        loop='Loop1_pressure_reinforcement', flow=flow_col, outcome='addiction_index_change',
        n_raw=len(sub), r_raw=round(r_raw,4), p_raw=round(p_raw,4),
        n_partial=n_part, r_partial=round(r_part,4), p_partial=round(p_part,4),
    ))
    print(f'{flow_col} ~ addiction_index_change: raw r={r_raw:.4f} p={p_raw:.4g} | '
          f'partial (control addiction_index_pre) r={r_part:.4f} p={p_part:.4g}')

---

## Section 3: Loop 2 -- Self-Regulation Correction (volume_change -> self_control_index_change)

In [ ]:
# =============================================================================
# Loop 2: volume_change predicting self_control_index_change, raw AND partial
# (controlling for self_control_index_pre)
# =============================================================================
sub2 = stock_flow[['volume_change', 'self_control_index_change', 'self_control_index_pre']].dropna()
r2_raw, p2_raw = stats.pearsonr(sub2['volume_change'], sub2['self_control_index_change'])
r2_part, p2_part, n2_part = partial_corr(
    stock_flow['volume_change'], stock_flow['self_control_index_change'], stock_flow['self_control_index_pre']
)
loop_results.append(dict(
    loop='Loop2_self_regulation_correction', flow='volume_change', outcome='self_control_index_change',
    n_raw=len(sub2), r_raw=round(r2_raw,4), p_raw=round(p2_raw,4),
    n_partial=n2_part, r_partial=round(r2_part,4), p_partial=round(p2_part,4),
))
print(f'volume_change ~ self_control_index_change: raw r={r2_raw:.4f} p={p2_raw:.4g} | '
      f'partial (control self_control_index_pre) r={r2_part:.4f} p={p2_part:.4g}')

feedback_loop_tests = pd.DataFrame(loop_results)
print()
print(feedback_loop_tests.to_string(index=False))

---

## Section 4: Regime x Stock and Condition x Stock Summaries

In [ ]:
# =============================================================================
# Regime x stock summary: does DyS baseline regime relate to psychosocial change?
# =============================================================================
regime_stock_rows = []
for regime_label in ['declining', 'growing', 'resting', 'stochastic']:
    sub = stock_flow[stock_flow['regime_baseline'] == regime_label]
    regime_stock_rows.append(dict(
        regime_baseline=regime_label, n=len(sub),
        addiction_index_change_mean=sub['addiction_index_change'].mean(),
        self_control_index_change_mean=sub['self_control_index_change'].mean(),
    ))
regime_stock_summary = pd.DataFrame(regime_stock_rows).round(4)
print(regime_stock_summary.to_string(index=False))

In [ ]:
# =============================================================================
# Condition x stock summary
# =============================================================================
condition_stock_rows = []
for cond in ['control', 'default', 'intervention']:
    sub = stock_flow[stock_flow['study_condition'] == cond]
    condition_stock_rows.append(dict(
        study_condition=cond, n=len(sub),
        addiction_index_change_mean=sub['addiction_index_change'].mean(),
        self_control_index_change_mean=sub['self_control_index_change'].mean(),
    ))
condition_stock_summary = pd.DataFrame(condition_stock_rows).round(4)
print(condition_stock_summary.to_string(index=False))

f_stat, p_anova = stats.f_oneway(*[
    stock_flow.loc[stock_flow.study_condition==c, 'addiction_index_change'].dropna()
    for c in ['control','default','intervention']
])
print(f'\nANOVA addiction_index_change ~ condition: F={f_stat:.3f}, p={p_anova:.4f}')

In [ ]:
# =============================================================================
# Figure 1: Loop 1 scatter (mean_N_baseline vs addiction_index_change)
# =============================================================================
fig, ax = plt.subplots(figsize=(7, 6))
sub = stock_flow[['mean_N_baseline', 'addiction_index_change']].dropna()
ax.scatter(sub['mean_N_baseline'], sub['addiction_index_change'], alpha=0.5, s=25)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
row = feedback_loop_tests[(feedback_loop_tests.loop=='Loop1_pressure_reinforcement') & (feedback_loop_tests.flow=='mean_N_baseline')].iloc[0]
ax.set_xlabel('mean_N_baseline (StoP, notebook 73)')
ax.set_ylabel('addiction_index_change')
ax.set_title(f'Loop 1: raw r={row.r_raw:.3f} (p={row.p_raw:.3g}), partial r={row.r_partial:.3f} (p={row.p_partial:.3g})')
plt.tight_layout()
fig.savefig(FIGURES / 'fig1_syd_loop1_scatter.png', dpi=150)
plt.show()
print('Saved fig1_syd_loop1_scatter.png')

In [ ]:
# =============================================================================
# Figure 2: Loop 2 scatter (volume_change vs self_control_index_change)
# =============================================================================
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(sub2['volume_change'], sub2['self_control_index_change'], alpha=0.5, s=25, color='#DD8452')
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('volume_change (StoP/NeT bridge quantity)')
ax.set_ylabel('self_control_index_change')
ax.set_title(f'Loop 2: raw r={r2_raw:.3f} (p={p2_raw:.3g}), partial r={r2_part:.3f} (p={p2_part:.3g})')
plt.tight_layout()
fig.savefig(FIGURES / 'fig2_syd_loop2_scatter.png', dpi=150)
plt.show()
print('Saved fig2_syd_loop2_scatter.png')

In [ ]:
# =============================================================================
# Figure 3: regime x stock comparison
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
order = ['declining', 'resting', 'growing', 'stochastic']
plot_df = regime_stock_summary.set_index('regime_baseline').reindex(order)
axes[0].bar(order, plot_df['addiction_index_change_mean'], color='#C44E52')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Mean addiction_index_change by baseline regime')
axes[0].set_ylabel('addiction_index_change')
axes[1].bar(order, plot_df['self_control_index_change_mean'], color='#55A868')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Mean self_control_index_change by baseline regime')
axes[1].set_ylabel('self_control_index_change')
plt.tight_layout()
fig.savefig(FIGURES / 'fig3_syd_regime_stock_comparison.png', dpi=150)
plt.show()
print('Saved fig3_syd_regime_stock_comparison.png')

In [ ]:
# =============================================================================
# Figure 4: raw vs partial correlation comparison (the regression-to-mean check)
# =============================================================================
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(feedback_loop_tests))
width = 0.35
ax.bar(x - width/2, feedback_loop_tests['r_raw'], width, label='raw r', color='#8172B2')
ax.bar(x + width/2, feedback_loop_tests['r_partial'], width, label='partial r (baseline-controlled)', color='#4C72B0')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f"{row.loop.replace('_',' ')}\n({row.flow})" for row in feedback_loop_tests.itertuples()], fontsize=8)
ax.set_ylabel('correlation coefficient')
ax.set_title('DS6 SyD -- raw vs partial correlation (regression-to-the-mean check)')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES / 'fig4_syd_raw_vs_partial_correlation.png', dpi=150)
plt.show()
print('Saved fig4_syd_raw_vs_partial_correlation.png')

In [ ]:
# =============================================================================
# Export all outputs
# =============================================================================
out_cols = ['participant_id_canonical', 'study_condition',
            'addiction_index_pre', 'addiction_index_post', 'addiction_index_change',
            'self_control_index_pre', 'self_control_index_post', 'self_control_index_change',
            'mean_N_baseline', 'mu_hat_baseline', 'volume_change',
            'regime_baseline', 'regime_intervention']
participant_stock_flow = stock_flow[out_cols].copy()
participant_stock_flow.to_csv(REPORTS / 'participant_stock_flow.csv', index=False)
print(f'Saved participant_stock_flow.csv ({len(participant_stock_flow)} rows)')

feedback_loop_tests.to_csv(REPORTS / 'feedback_loop_tests.csv', index=False)
print(f'Saved feedback_loop_tests.csv ({len(feedback_loop_tests)} rows)')

regime_stock_summary.to_csv(REPORTS / 'regime_stock_summary.csv', index=False)
print(f'Saved regime_stock_summary.csv ({len(regime_stock_summary)} rows)')

condition_stock_summary.to_csv(REPORTS / 'condition_stock_summary.csv', index=False)
print(f'Saved condition_stock_summary.csv ({len(condition_stock_summary)} rows)')

In [ ]:
# =============================================================================
# model_run_summary.md
# =============================================================================
loop1 = feedback_loop_tests[feedback_loop_tests.loop=='Loop1_pressure_reinforcement']
loop2 = feedback_loop_tests[feedback_loop_tests.loop=='Loop2_self_regulation_correction'].iloc[0]

summary_md = f"""# DS6 SyD -- Model Run Summary

**Run at:** {MODEL_RUN_TS}
**Notebook:** notebooks/78_ds6_system_dynamics.ipynb
**Blueprint:** outputs/reports/stage4/step1_modeling/ds6_syd_blueprint.md

## Sample
- Prepost-eligible participants (both survey waves + behavioral data): {len(stock_flow)}

## Loop 1 -- pressure reinforcement (baseline attempt volume -> addiction_index_change)
{loop1.to_string(index=False)}

## Loop 2 -- self-regulation correction (volume_change -> self_control_index_change)
raw r={r2_raw:.4f} (p={p2_raw:.4g}, n={len(sub2)}) | partial r={r2_part:.4f} (p={p2_part:.4g}, n={n2_part})

## Condition-level ANOVA (addiction_index_change ~ study_condition)
F={f_stat:.3f}, p={p_anova:.4f}

## Proxy warning
addiction_index/self_control_index are self-reported psychosocial scales. Loop test results
describe statistical association between behavioral flow and self-reported change, not a
verified mechanistic causal pathway.

## Known limitations
- N=188 (prepost-eligible) — smaller than the full N=243 behavioral sample.
- Regression-to-the-mean confound explicitly tested (raw vs partial correlation) per
  blueprint Part 3 — only the partial (baseline-level-controlled) result is treated as
  evidence for either loop.
- study_condition is quasi-experimental (cohort-level), not individually randomized.
- Only 2 survey waves — this is NOT a repeated weekly panel like Structured's SyD; the
  person-demeaning technique used there does not directly apply (a 2-point series demeaned
  IS the change score, already used as the outcome here).
"""

(REPORTS / 'model_run_summary.md').write_text(summary_md)
print(summary_md)

---

## Section 5: Interpretation and Limits

### Headline finding: both feedback loops are genuine null results

Neither loop hypothesis is supported, before or after the regression-to-the-mean control.
Loop 1 partial r=0.057 (p=0.45) / -0.111 (p=0.14); Loop 2 partial r=0.093 (p=0.22).
Condition ANOVA on addiction_index_change also non-significant (F=1.225, p=0.296).
Reported as a genuine null, not reframed -- consistent with the mostly-null psychosocial
pattern already seen across every other DS6 Step 1 notebook.

### What the SyD model reveals

| Result | Interpretation |
|---|---|
| Loop 1 (raw vs partial) | Whether baseline attempt volume predicts worsening self-reported addiction, beyond what baseline severity alone would predict via regression to the mean |
| Loop 2 (raw vs partial) | Whether cutting back on attempt volume predicts improving self-reported self-control, beyond baseline-severity regression to the mean |
| Regime x stock | Whether DyS's behavioral regime classification (declining/growing/resting/stochastic) relates to psychosocial change |

### What this model does NOT show

- A verified causal mechanism -- correlational, not experimental (condition is
  quasi-experimental, not individually randomized).
- That self-reported change reflects a "real" psychological shift rather than reporting
  artifacts, social desirability, or measurement noise.
- Generalizability beyond the N=188 prepost-eligible subset (60 of 243 behavioral
  participants lack a wave-2 survey).

### Forward connections (informational only -- not implemented here)

- **SyD -> Step 2 phenotyping:** the loop-test partial correlations and regime x stock
  summary are candidate phenotype dimensions once DS6 Step 1 concludes (StM remains; GaT/ABM
  deferred).
- **SyD closes the repeated-wave rationale** that specifically justified deferring this
  model family from DS5 to DS6 in the project's own sequencing plan.